# Find ocean-icechunks datasets by place and time

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ocean-icechunks/2026-ioos-hackathon/blob/main/notebooks/stac-geoparquet-query.ipynb)

Which Icechunk stores cover a place and a period? We search the STAC geoparquet catalog built
by `stac-geoparquet-create` with [rustac](https://github.com/stac-utils/rustac), then open
the stores that match.

**Run `stac-geoparquet-create` first, in the same session.** It writes the catalog file that
this notebook reads.

In [ ]:
%pip install -q "icechunk>=2.2" "xarray>=2026.7" "zarr>=3.4" rustac

In [ ]:
from pathlib import Path

PARQUET = "ocean-icechunks-items.parquet"
if not Path(PARQUET).exists():
    raise FileNotFoundError(f"{PARQUET} not found: run stac-geoparquet-create first, in this same session")

## 1. The whole catalog

With no filter, a search returns every item.

In [ ]:
import rustac

everything = await rustac.search(PARQUET)
print(len(everything), "datasets")
for item in everything:
    p = item["properties"]
    print(f"{item['id']:36s} {p.get('start_datetime', 'no time axis')[:10]}  {p.get('end_datetime', '')[:10]}")

## 2. Search by place and time

Oahu, Hawaii, during January 2026. The `bbox` is west, south, east, north in degrees. Items whose
box crosses the antimeridian (the North Pacific ocean heat content) are matched correctly.

In [ ]:
BBOX = [-158.5, 20.5, -157.5, 21.5]
DATETIME = "2026-01-01/2026-02-01"

hits = await rustac.search(PARQUET, bbox=BBOX, datetime=DATETIME)
print(f"{len(hits)} of {len(everything)} datasets intersect the box during {DATETIME}\n")
for item in hits:
    p, a = item["properties"], item["assets"]["icechunk"]
    print(item["id"])
    print(f"  {p['start_datetime']} .. {p['end_datetime']}")
    print(f"  {a['href']}  (group: {a['icechunk:group']})")

assert 0 < len(hits) < len(everything), "expected some, but not all, datasets"

Several datasets match, but not all. The reanalyses ended before 2026, the ocean heat content
stores for other ocean basins do not cover Hawaii, and so on. Try your own place and dates by
changing `BBOX` and `DATETIME`.

## 3. Open a match

The asset tells us where the store is and which group to open.

In [ ]:
import icechunk
import xarray as xr
from urllib.parse import urlparse

item = hits[0]
asset = item["assets"]["icechunk"]
url = urlparse(asset["href"])
storage = icechunk.s3_storage(bucket=url.netloc, prefix=url.path.lstrip("/"), region="us-west-2", anonymous=True)
store = icechunk.Repository.open(storage).readonly_session(asset["icechunk:branch"]).store
group = asset["icechunk:group"]
ds = xr.open_zarr(store, group=None if group == "/" else group, consolidated=False)
print(item["id"])
ds